# Lab 4 — Performance and Spark internals

**Objectives**
- Understand jobs, stages and tasks, and what a shuffle is
- Read an execution plan with `explain()`
- Choose the right join strategy (broadcast vs sort-merge)
- Detect and fix *data skew*
- Know what AQE does automatically
- Decide when to use cache and when to avoid UDFs

---
## 1. Setup

In [1]:
# !pip install -q pyspark==4.0.1 pyarrow

In [2]:
import io, re, time, contextlib
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Lab 4 - Performance")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "false")   # off on purpose; we turn it on in Lab 5
    .config("spark.ui.showConsoleProgress", "false") # hides the progress bars
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Cores   :", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:19:27 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:19:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:19:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark   : 4.0.1
Cores   : 24
Spark UI: http://10.23.20.50:4040


Helper functions used in this notebook:

- `timer` measures time and gives the job a name (it shows up in the Spark UI).
- `run` forces execution **without bringing data to the driver**, using the `noop` format. This is the correct way to measure performance: `show()` reads only a few rows and `collect()` also measures the transfer to Python.
- `count_shuffles` counts how many `Exchange` operators (shuffles) are in the plan.

In [3]:
@contextlib.contextmanager
def timer(label):
    spark.sparkContext.setJobDescription(label)
    t0 = time.perf_counter()
    yield # when it finishes
    print(f"{label:<45} {time.perf_counter() - t0:6.2f} s")
    spark.sparkContext.setJobDescription(None)

def run(df):
    df.write.format("noop").mode("overwrite").save()

def plan_text(df, mode="simple"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

def count_shuffles(df):
    return len(re.findall(r"\bExchange\b", plan_text(df)))

def partition_balance(df, *keys, n=16, label=""):
    """Repartitions by key and shows the size of the largest partition vs the median."""
    s = (df.repartition(n, *keys)
           .groupBy(F.spark_partition_id()).count()
           .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
           .first())
    print(f"{label:<12} largest = {s['max']:>9,} | median = {s['median']:>9,} | ratio = {s['max'] / s['median']:.1f}x")

---
## 2. A large dataset

We will generate three tables in the style of Lab 2, but at scale:

| Table | Rows | Role |
|--------|--------|-------|
| `orders` | 5 million | Facts (orders) |
| `customers` | 200 thousand | Medium dimension |
| `products` | 1 thousand | Small dimension |

`spark.range()` generates data that is already distributed, without going through Python. We save it as Parquet so that later reads are real I/O.

> If your machine is modest (or on free Colab), reduce `N_ORDERS` to 2 million.

In [4]:
N_ORDERS    = 5_000_000
N_CUSTOMERS = 200_000
N_PRODUCTS  = 1_000
REGIONS     = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES  = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
BASE        = "/tmp/lab4"

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

orders_gen = (
    spark.range(N_ORDERS).withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.rand(1) * N_CUSTOMERS).cast("int"))
    .withColumn("product_id",  (F.rand(2) * N_PRODUCTS).cast("int"))
    .withColumn("quantity",    (F.rand(3) * 5 + 1).cast("int"))
    .withColumn("price",       F.round(F.rand(4) * 490 + 10, 2))
    .withColumn("region",      pick(REGIONS, 5))
    .withColumn("order_date",  F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand(6) * 365).cast("int")))
)

customers_gen = (
    spark.range(N_CUSTOMERS).withColumnRenamed("id", "customer_id")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("segment", pick(["Consumer", "Corporate", "Small Business"], 11))
    .withColumn("signup_year", (F.rand(12) * 6 + 2019).cast("int"))
)

products_gen = (
    spark.range(N_PRODUCTS).withColumnRenamed("id", "product_id")
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("product_id")))
    .withColumn("category", pick(CATEGORIES, 21))
)

with timer("Generate and save the 3 tables"):
    orders_gen.write.mode("overwrite").parquet(f"{BASE}/orders")
    customers_gen.write.mode("overwrite").parquet(f"{BASE}/customers")
    products_gen.write.mode("overwrite").parquet(f"{BASE}/products")

orders    = spark.read.parquet(f"{BASE}/orders")
customers = spark.read.parquet(f"{BASE}/customers")
products  = spark.read.parquet(f"{BASE}/products")

orders.show(5)

Generate and save the 3 tables                  2.52 s
+--------+-----------+----------+--------+------+---------+----------+
|order_id|customer_id|product_id|quantity| price|   region|order_date|
+--------+-----------+----------+--------+------+---------+----------+
| 2291666|      73092|       417|       1| 98.89|  Midwest|2024-12-19|
| 2291667|     186251|       989|       3|317.22|    North|2024-01-28|
| 2291668|     140692|       483|       4|183.57|    North|2024-01-20|
| 2291669|      32718|       607|       5| 113.4|    North|2024-07-17|
| 2291670|       1265|       401|       5|411.26|Northeast|2024-04-30|
+--------+-----------+----------+--------+------+---------+----------+
only showing top 5 rows


### 2.1 Pandas vs Spark with 5 million rows

Same query in both tools: total revenue by region.
The result is often surprising.

In [5]:
with timer("Pandas: read Parquet + groupby"):
    pdf = pd.read_parquet(f"{BASE}/orders")
    pdf["revenue"] = pdf["price"] * pdf["quantity"]
    res_pd = pdf.groupby("region")["revenue"].sum()

with timer("Spark: read Parquet + groupBy"):
    res_sp = (
        spark.read.parquet(f"{BASE}/orders")
        .groupBy("region")
        .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
        .collect()
    )

print(f"\nMemory used by the Pandas DataFrame: {pdf.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
del pdf

Pandas: read Parquet + groupby                  0.66 s
Spark: read Parquet + groupBy                   0.63 s

Memory used by the Pandas DataFrame: 660 MB


**How to read this:** on a single machine, with data that fits in RAM, Pandas often ties or wins. Spark pays a fixed cost (planning, JVM, task coordination).

Spark pays off when:
- the data **does not fit in the memory** of one machine (Pandas simply breaks);
- there is a **cluster**: the same code scales to dozens of machines without changes;
- the pipeline reads from distributed sources (data lake, S3, HDFS).

That is why Spark performance is less about "which is faster" and more about **"how to avoid unnecessary work"**. That is what the rest of this lab covers.

---
## 3. How Spark executes: job, stage and task

When you call an **action**, Spark turns the plan into physical work:

| Concept | What it is | How many |
|----------|---------|---------|
| **Job** | Work triggered by an action (`count`, `write`, `collect`) | 1 per action (sometimes more) |
| **Stage** | Block of operations that run without exchanging data between partitions | New stage at each shuffle |
| **Task** | Execution of a stage on **one** partition | 1 per partition |

```
  df.filter(...).withColumn(...).groupBy("region").sum()  .write  <- action
                                                            │
                                                           JOB
            ┌───────────────────────────────────────────────┴────────────┐
         STAGE 1                                                       STAGE 2
   read + filter + withColumn                                        final sum
   + partial sum per partition                                       by region
   [T1] [T2] [T3] [T4]   ─────────── SHUFFLE (Exchange) ──────────▶  [T1] [T2] [T3]
```

### Narrow vs wide transformations

| Type | Each output partition depends on... | Examples | Shuffle? |
|------|-------------------------------------|----------|----------|
| **Narrow** | one input partition | `select`, `filter`, `withColumn`, `union` | No |
| **Wide** | several input partitions | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` | Yes |

```
   NARROW                          WIDE (shuffle)
   P1 ──▶ P1'                      P1 ──┬──▶ P1'
   P2 ──▶ P2'                      P2 ──┼──▶ P2'
   P3 ──▶ P3'                      P3 ──┴──▶ P3'
```

The **shuffle** is Spark's most expensive operation: data is serialized, written to disk, sent over the network and read again. **Optimizing Spark is, to a large extent, reducing shuffles and the volume that goes through them.**

### 3.1 Reading the plan with `explain()`

In the physical plan, every shuffle appears as an **`Exchange`**. The plan is read **from bottom to top**.

In [6]:
query = (
    orders
    .filter(F.col("price") > 100)                                 # narrow
    .withColumn("revenue", F.col("price") * F.col("quantity"))    # narrow
    .groupBy("region")                                            # wide
    .agg(F.sum("revenue").alias("revenue"))
)

query.explain(mode="formatted")
print("Shuffles in the plan:", count_shuffles(query))

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [quantity#21, price#22, region#23]
Batched: true
Location: InMemoryFileIndex [file:/tmp/lab4/orders]
PushedFilters: [IsNotNull(price), GreaterThan(price,100.0)]
ReadSchema: struct<quantity:int,price:double,region:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]

(3) Filter [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]
Condition : (isnotnull(price#22) AND (price#22 > 100.0))

(4) Project [codegen id : 1]
Output [2]: [region#23, (price#22 * cast(quantity#21 as double)) AS revenue#74]
Input [3]: [quantity#21, price#22, region#23]

(5) HashAggregate [codegen id : 1]
Input [2]: [region#23, revenue#74]
Keys [1]: [region#23]
Functions [1]: [partial_sum(revenue#74)]
Aggregate Attributes [1]: [sum#85]
Results

Things to look for in the plan:

- **`FileScan parquet`** with `PushedFilters`: the `price > 100` filter was pushed down to the read (*predicate pushdown*).
- **`ReadSchema`**: only the columns used are read (*column pruning*). An advantage of Parquet, which is columnar.
- **`HashAggregate (partial)`** before the `Exchange`: each partition sums locally, and only the partial totals cross the shuffle.
- **`Exchange hashpartitioning(region, 200)`**: the shuffle, with 200 output partitions.

In [7]:
# A query with sort + aggregation + join: how many shuffles?
q2 = (
    orders
    .join(products, "product_id")
    .groupBy("category").agg(F.count("*").alias("n"))
    .orderBy(F.desc("n"))
)
print("Shuffles:", count_shuffles(q2))
q2.explain()

Shuffles: 2
== Physical Plan ==
*(4) Sort [n#89L DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(n#89L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=255]
   +- *(3) HashAggregate(keys=[category#30], functions=[count(1)])
      +- Exchange hashpartitioning(category#30, 200), ENSURE_REQUIREMENTS, [plan_id=251]
         +- *(2) HashAggregate(keys=[category#30], functions=[partial_count(1)])
            +- *(2) Project [category#30]
               +- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false
                  :- *(2) Filter isnotnull(product_id#20)
                  :  +- *(2) ColumnarToRow
                  :     +- FileScan parquet [product_id#20] Batched: true, DataFilters: [isnotnull(product_id#20)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/lab4/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<product_id:int>
                  +- BroadcastExchange HashedRelationBroadca

---
## 4. Partitions

The partition is the unit of parallelism: **one task processes one partition**. Too few partitions leave cores idle. Too many generate scheduling overhead and lots of small files.

| Moment | Who decides the number of partitions | Setting |
|---------|----------------------------------|--------------|
| Reading files | File size | `spark.sql.files.maxPartitionBytes` (128 MB) |
| After a shuffle | Fixed setting (without AQE) | `spark.sql.shuffle.partitions` (200) |
| Manual | You | `repartition(n)` / `coalesce(n)` |

Rule of thumb: partitions between **100 MB and 200 MB**, and at least **2 to 4 times the number of cores**.

In [8]:
print("Partitions on read:", orders.rdd.getNumPartitions())

# How many rows are in each partition?
(
    orders
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show()
)

Partitions on read: 24
+---------+------+
|partition| count|
+---------+------+
|        0|208334|
|        1|208333|
|        2|208333|
|        3|208333|
|        4|208333|
|        5|208333|
|        6|208333|
|        7|208334|
|        8|208334|
|        9|208333|
|       10|208333|
|       11|208334|
|       12|208333|
|       13|208333|
|       14|208333|
|       15|208333|
|       16|208333|
|       17|208334|
|       18|208334|
|       19|208333|
+---------+------+
only showing top 20 rows


In [9]:
# Effect of spark.sql.shuffle.partitions on an aggregation
agg = orders.groupBy("customer_id").agg(F.sum("price").alias("total"))

for n in [200, 2, spark.sparkContext.defaultParallelism * 2]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    with timer(f"groupBy with shuffle.partitions = {n}"):
        run(agg)

spark.conf.set("spark.sql.shuffle.partitions", 200)

groupBy with shuffle.partitions = 200           0.89 s
groupBy with shuffle.partitions = 2             0.57 s
groupBy with shuffle.partitions = 48            0.21 s


With small data, 200 partitions generate 200 tiny tasks, and the overhead dominates. With 2, a few tasks do all the work and the cores sit idle. The ideal value depends on the volume, which is exactly what AQE adjusts on its own (Lab 5).

### `repartition` vs `coalesce`

In [10]:
print("repartition(8):")
orders.repartition(8).explain()

print("coalesce(2):")
orders.coalesce(2).explain()

repartition(8):
== Physical Plan ==
Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=466]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/lab4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...


coalesce(2):
== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/lab4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...




| | `repartition(n)` | `coalesce(n)` |
|-|------------------|---------------|
| Shuffle | Yes (`Exchange`) | No |
| Can increase partitions | Yes | No, it only reduces |
| Distribution | Even | May become unbalanced |
| Typical use | Before heavy joins, by key | Reducing files before writing |

---
## 5. Join strategies

A join needs rows with the same key to be on the same machine. There are two main ways to achieve this:

```
  SORT-MERGE JOIN (default)                BROADCAST HASH JOIN
  both tables go through a shuffle          the small table is copied to all

  orders ──shuffle──┐                       products (small)
                    ├── sort + merge          │ copy  │ copy  │ copy
  customers ─shuffle┘                       [orders P1] [orders P2] [orders P3]
                                            no shuffle of the large table
```

| Strategy | When to use | Cost |
|------------|-------------|-------|
| **Broadcast Hash Join** | One side is small (default: < 10 MB) | No shuffle on the large side; the small side goes into each executor's memory |
| **Sort-Merge Join** | Two large tables | Shuffle + sort on both sides |
| **Shuffle Hash Join** | One medium side, no sort | Shuffle on both sides, hash of the smaller one |

Spark chooses broadcast automatically below `spark.sql.autoBroadcastJoinThreshold`. Let's turn that limit off to compare.

In [11]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # turns off automatic broadcast

smj = orders.join(products, "product_id")
print("Sort-merge — shuffles:", count_shuffles(smj))
with timer("Join orders x products (sort-merge)"):
    run(smj)

bhj = orders.join(F.broadcast(products), "product_id")
print("\nBroadcast  — shuffles:", count_shuffles(bhj))
with timer("Join orders x products (broadcast)"):
    run(bhj)

Sort-merge — shuffles: 2
Join orders x products (sort-merge)             1.05 s

Broadcast  — shuffles: 0
Join orders x products (broadcast)              0.37 s


In [12]:
# Confirm in the plan which strategy was used
print([l.strip() for l in plan_text(smj).splitlines() if "Join" in l])
print([l.strip() for l in plan_text(bhj).splitlines() if "Join" in l])

['+- *(5) SortMergeJoin [product_id#20], [product_id#28], Inner']
['+- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false']


Three equivalent ways to ask for a broadcast:

```python
orders.join(F.broadcast(products), "product_id")      # function
orders.join(products.hint("broadcast"), "product_id") # API hint
spark.sql("SELECT /*+ BROADCAST(p) */ * FROM orders o JOIN products p USING (product_id)")
```

> Careful: broadcasting a large table can exhaust the memory of the driver and the executors. Use it for small dimensions.

In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10MB")   # restores the default

In [14]:
spark.stop()